# 6 · Putting the rule to the test

Companion to the tutorial *From Hand-Crafted to LLM-Based Variation Operators in Metaheuristics*. It runs offline: the model is a fixed pool of completions, so every number below comes out the same on your machine.

The placement rule says: remove a channel and see whether the proposals change.
That is a testable claim, so it should be tested, and this notebook runs the test
on the natural-language channel of an EoH-style operator for bin packing.

The answer came back negative. It is reported here for what it is, and the second
half of the notebook works out why, which turns out to be more interesting than a
positive result would have been.

In [1]:
import _bootstrap
import json, subprocess, sys
import viz
import bpp_amortized as bpp
from llm import envelope

candidates = json.load(open('candidates.json'))
print({k: len(v) for k, v in candidates.items()})

{'_comment': 485, 'FULL': 3, 'ABLATED': 3}


## The two conditions

**FULL** holds heuristics generated with the parent code, the scores, and the
natural-language idea: *prefer the bin with the least remaining slack*.

**ABLATED** holds heuristics generated with the parent code and the scores, and
nothing else. Same model, same evaluator, same instances. One channel removed.

In [2]:
for cond in ('FULL', 'ABLATED'):
    print(f'═══ {cond} ═══')
    for code_text in candidates[cond]:
        print(code_text.strip())
        print('  ·')

═══ FULL ═══
def priority(item, bins):
    # least remaining slack after placing (best-fit)
    return [-(b - item) for b in bins]
  ·
def priority(item, bins):
    # same tightest-fit rule, written as item - b
    return [item - b for b in bins]
  ·
def priority(item, bins):
    # tightest fit by absolute slack
    return [-abs(b - item) for b in bins]
  ·
═══ ABLATED ═══
def priority(item, bins):
    # smallest remaining capacity
    return [-b for b in bins]
  ·
def priority(item, bins):
    # prefer the fullest open bin (capacity 10)
    return [10 - b for b in bins]
  ·
def priority(item, bins):
    # prefer low remaining capacity (monotone in b)
    return [-(b * b) for b in bins]
  ·


## Scoring them the way the paper does

In [3]:
print(subprocess.run([sys.executable, 'ablation.py', 'candidates.json'],
                     capture_output=True, text=True).stdout)

online BPP lower bound = 19 bins; parent first-fit gap = 15.8%

[FULL]  n=3  valid=3
  gap%: mean=0.0  min=0.0  max=0.0  optimal(gap=0)=3/3
  implements least-slack idea: 3/3

[ABLATED]  n=3  valid=3
  gap%: mean=0.0  min=0.0  max=0.0  optimal(gap=0)=3/3
  implements least-slack idea: 0/3



In [4]:
LB = bpp.lower_bound_total()
gaps = {c: [100.0 * (bpp.evaluator(bpp.parse(envelope(t))) - LB) / LB
            for t in candidates[c]] for c in ('FULL', 'ABLATED')}

viz.compare([('with the idea', gaps['FULL']), ('without it', gaps['ABLATED'])],
            'Drop-channel ablation: gap of each emitted heuristic')

'<svg xmlns="http://www.w3.org/2000/svg" width="620" height="216" viewBox="0 0 620 216" font-family="ui-sans-serif,-apple-system,Segoe UI,Roboto,sans-serif"><rect width="620" height="216" fill="#ffffff"/><text x="20.0" y="20.0" font-size="13" fill="#1f2430" text-anchor="start" font-weight="600">Drop-channel ablation: gap of each emitted heuristic</text><text x="106.0" y="70.0" font-size="11" fill="#1f2430" text-anchor="end" font-weight="600">with the idea</text><line x1="118" y1="66" x2="590" y2="66" stroke="#d8dce3" stroke-width="1"/><circle cx="118.0" cy="66" r="6" fill="#1f9d78" fill-opacity="0.8"/><circle cx="118.0" cy="53" r="6" fill="#1f9d78" fill-opacity="0.8"/><circle cx="118.0" cy="40" r="6" fill="#1f9d78" fill-opacity="0.8"/><text x="118.0" y="85.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">3 candidates</text><text x="106.0" y="136.0" font-size="11" fill="#1f2430" text-anchor="end" font-weight="600">without it</text><line x1="118" y1="132" x2="590" y2="132" stroke="#d8dce3" stroke-width="1"/><circle cx="118.0" cy="132" r="6" fill="#1f9d78" fill-opacity="0.8"/><circle cx="118.0" cy="119" r="6" fill="#1f9d78" fill-opacity="0.8"/><circle cx="118.0" cy="106" r="6" fill="#1f9d78" fill-opacity="0.8"/><text x="118.0" y="151.0" font-size="10" fill="#8a8f9a" text-anchor="start" font-weight="normal">3 candidates</text><line x1="118" y1="190" x2="590" y2="190" stroke="#d8dce3" stroke-width="1"/><text x="118.0" y="206.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">0</text><text x="590.0" y="206.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">10</text><text x="354.0" y="206.0" font-size="10" fill="#8a8f9a" text-anchor="middle" font-weight="normal">gap %</text></svg>'

Every candidate is optimal, in both conditions. On this task the natural-language
channel is not load-bearing: the operator recovers the best-fit rule with the idea
and without it.

The one thing that did change is the wording of what came out. The ablated
heuristics stopped mentioning slack; they just did not need the idea to arrive at
the same rule.

## Why the test could not have said much

A measurement can only show a difference if there is room for one. Here there was
almost none, and it is worth seeing that in numbers rather than taking it on
faith.

In [5]:
import itertools

rules = {
    'first fit': 'def priority(item, bins):\n    return [-i for i in range(len(bins))]',
    'best fit':  'def priority(item, bins):\n    return [-(b - item) for b in bins]',
    'worst fit': 'def priority(item, bins):\n    return [b - item for b in bins]',
    'random-ish':'def priority(item, bins):\n    return [(7 * i) % 5 for i in range(len(bins))]',
}
print(f'lower bound: {LB} bins\n')
for name, src in rules.items():
    total = bpp.evaluator(bpp.parse(envelope(src)))
    print(f'{name:11s} {total:3d} bins   gap {100.0 * (total - LB) / LB:5.1f}%')

lower bound: 19 bins

first fit    22 bins   gap  15.8%
best fit     19 bins   gap   0.0%
worst fit    22 bins   gap  15.8%
random-ish   21 bins   gap  10.5%


Even a deliberately silly rule lands close to the bound. These instances are small
and their items pack cleanly, so the distance between a good heuristic and a poor
one is a couple of bins. An ablation run on a task like this cannot separate the
two conditions, whatever the channel does.

So the honest reading has two parts, and the second is the useful one.

1. On this task, removing the natural-language channel changed nothing measurable.
2. This task was never going to detect a change, so statement 1 is weak evidence
   about the channel and strong evidence about the instance set.

## What a sharper test needs

Three properties, none of them exotic.

- **Headroom.** Instances where first fit and best fit come apart by a lot, so a
  better rule has somewhere to show up.
- **Enough candidates.** Three per condition measures nothing. The comparison is
  between distributions of emitted programs, and three points do not make one.
- **Everything else held fixed.** Same parent code, same scores, same evaluator,
  same decoding settings. The channel is the only thing that moves.

The harness is already here and it takes a JSON file, so running a better version
is a matter of generating candidates rather than writing code.

In [6]:
template = {'FULL': ['def priority(item, bins): ...'],
            'ABLATED': ['def priority(item, bins): ...']}
print(json.dumps(template, indent=2))
print('\nsave as my_candidates.json, then:')
print('  uv run ablation.py my_candidates.json')

{
  "FULL": [
    "def priority(item, bins): ..."
  ],
  "ABLATED": [
    "def priority(item, bins): ..."
  ]
}

save as my_candidates.json, then:
  uv run ablation.py my_candidates.json


## Try it

1. Build an instance set where first fit is 30% above the bound. Re-run the four
   rules above and watch the spread open up.
2. Generate ten candidates per condition with a real model through `AnthropicLLM`,
   pinning the snapshot and the seed, and run the ablation on those.
3. Ablate a different channel. Remove the scores instead of the idea, and keep the
   code and the prose. Which removal hurts more is the placement rule doing its
   job.

A result that differs from the appendix would be a finding about the paper, so
write down what you expected before you run it.

---

Back to [the beginning](01_anatomy.ipynb), or to the
[repository README](../README.md).